# Nim Game with Minimax (Depth = 3)
This notebook implements a **two-player Nim game** where:
- Human vs Computer.
- Computer uses **Minimax Algorithm** with fixed depth = 3.
- Players can remove 1 to 3 objects from one pile per turn.
- The player who cannot make a move **loses**.


In [40]:
# Nim Game with Minimax (Fixed Depth 3)

from typing import List, Tuple
from dataclasses import dataclass
import time

@dataclass
class GameConfig:
    initial_piles: List[int]
    max_objects_per_move: int
    minimax_depth: int

    @classmethod
    def default(cls):
        # Default game configuration: 3 piles with 5 objects each,
        # maximum 3 objects can be removed per move, and minimax depth fixed at 3.
        
        return cls(
            initial_piles=[5, 5, 5],
            max_objects_per_move=3,
            minimax_depth=3  # Fixed depth 3 for assignment
        )


In [41]:
class GameState:
    def __init__(self, piles: List[int]):
        # Initialize game state with a list representing objects in each pile
        self.piles = piles.copy()

    def copy(self):
        # Create a copy of the current state to maintain immutability during moves
        return GameState(self.piles)

    def is_terminal(self):
        # Check if the game is over i.e. all piles are empty
        return all(pile == 0 for pile in self.piles)

    def get_valid_moves(self, max_objects: int) -> List[Tuple[int, int]]:
        # Generate all valid moves (pile index and number of objects to remove)
        
        moves = []
        for pile_idx, pile_size in enumerate(self.piles):
            # Objects removed can be from 1 up to max allowed or the pile size, whichever is smaller
            
            for objects in range(1, min(max_objects+1, pile_size+1)):
                moves.append((pile_idx, objects))
        return moves

    def apply_move(self, pile_idx: int, objects: int):
        # Apply a move and return new state without mutating current state
        
        new_state = self.copy()
        new_state.piles[pile_idx] -= objects
        return new_state

    def calculate_nim_sum(self) -> int:
        # Calculate nim-sum (XOR of all pile sizes), fundamental for Nim strategy
        
        nim_sum = 0
        for pile in self.piles:
            nim_sum ^= pile
        return nim_sum
        

In [42]:
class NimEvaluator:
    @staticmethod
    def evaluate(state: GameState, is_maximizing: bool, depth_remaining: int) -> float:
        # Static method to evaluate a game state’s desirability
        # Terminal states get high positive (win) or negative (lose) scores
        
        if state.is_terminal():
            return -1000 if is_maximizing else 1000
        nim_sum = state.calculate_nim_sum()
        
        # Positions with non-zero nim_sum are winning positions
        return 50 if nim_sum != 0 else -50
        

In [43]:
class MinimaxEngine:
    def __init__(self, config: GameConfig):
        # Constructor saves the configuration of the game for use in minimax
        self.config = config

    def get_best_move(self, state: GameState) -> Tuple[int, int]:
        # Finds the best next move by evaluating valid moves with minimax algorithm        
        best_move = None
        best_score = float('-inf')
        valid_moves = state.get_valid_moves(self.config.max_objects_per_move)
        for pile_idx, objects in valid_moves:
            new_state = state.apply_move(pile_idx, objects)
            
            # Recursive minimax call with depth reduced and alternate player turn
            score = self._minimax(
                new_state,
                self.config.minimax_depth - 1,
                False  # machine just moved, now it's human turn
            )
            if score > best_score:
                best_score = score
                best_move = (pile_idx, objects)
        return best_move

    def _minimax(self, state: GameState, depth: int, is_maximizing: bool) -> float:
        # Recursive helper for minimax algorithm evaluating states to a certain depth
        if depth == 0 or state.is_terminal():
            return NimEvaluator.evaluate(state, is_maximizing, depth)
        valid_moves = state.get_valid_moves(self.config.max_objects_per_move)
        
        if is_maximizing:
            # Maximize score in maximizing player turn
            max_eval = float('-inf')
            for pile_idx, objects in valid_moves:
                new_state = state.apply_move(pile_idx, objects)
                eval_score = self._minimax(new_state, depth - 1, False)
                max_eval = max(max_eval, eval_score)
            return max_eval
        else:
            # Minimize score in minimizing player turn
            min_eval = float('inf')
            for pile_idx, objects in valid_moves:
                new_state = state.apply_move(pile_idx, objects)
                eval_score = self._minimax(new_state, depth - 1, True)
                min_eval = min(min_eval, eval_score)
            return min_eval


In [44]:
class NimGame:
    def __init__(self, config=None):
        # Initialize game with configuration, starting state and minimax engine
        self.config = config or GameConfig.default()
        self.state = GameState(self.config.initial_piles)
        self.engine = MinimaxEngine(self.config)

    def display_state(self):
        # Display the current piles visually using stars and nim-sum for info
        print("\nCurrent piles:", self.state.piles)
        for i, pile in enumerate(self.state.piles):
            print(f"Pile {i+1}: {'*' * pile} ({pile})")
        print(f"Nim-sum: {self.state.calculate_nim_sum()}")

    def is_valid_move(self, pile_idx: int, objects: int) -> bool:
        # Validate a human player's intended move for legality
        if pile_idx < 0 or pile_idx >= len(self.state.piles):
            return False
        if objects < 1 or objects > self.config.max_objects_per_move:
            return False
        if objects > self.state.piles[pile_idx]:
            return False
        return True

    def get_human_move(self) -> Tuple[int, int]:
        # Prompt human player until a valid move is entered
        while True:
            try:
                pile_num = int(input(f"Enter pile number (1-{len(self.state.piles)}): "))
                pile_idx = pile_num - 1
                objects = int(input(f"Enter objects to remove (1-{self.config.max_objects_per_move}): "))
                if self.is_valid_move(pile_idx, objects):
                    return pile_idx, objects
                else:
                    print("Invalid move! Please try again.")
                    print(f"Pile {pile_num} has {self.state.piles[pile_idx]} objects.")
            except (ValueError, IndexError):
                print("Please enter valid numbers!")

    def play_turn(self, is_human: bool) -> bool:
        # Executes a turn for human or machine player and check if game continues
        if self.state.is_terminal():
            return False
        if is_human:
            print("\n" + "="*50)
            print("Your turn!")
            self.display_state()
            pile_idx, objects = self.get_human_move()
            self.state = self.state.apply_move(pile_idx, objects)
            print(f"\nYou removed {objects} objects from pile {pile_idx + 1}")
        else:
            print("\n" + "="*50)
            print("Machine's turn...")
            self.display_state()
            pile_idx, objects = self.engine.get_best_move(self.state)
            self.state = self.state.apply_move(pile_idx, objects)
            print(f"Machine removed {objects} objects from pile {pile_idx + 1}")
        return not self.state.is_terminal()

    def play_game(self):
        # Main game loop alternating human and machine turns until game ends
        print("WELCOME TO NIM GAME!")
        print("• The player who cannot make a move (all piles empty) loses")
        print("• Strategy tip: Watch the Nim-sum (XOR of pile sizes)!")
        is_human_turn = True
        while self.play_turn(is_human_turn):
            is_human_turn = not is_human_turn
        print("\n" + "="*60)
        print("GAME OVER!")
        self.display_state()
        # The player who just moved won (since the other player cannot move)
        if is_human_turn:
            print("Machine wins! The Machine outsmarted you this time.")
        else:
            print("Congratulations! You defeated the Machine!")
        print("="*60)
        
def run_nim_game():
    game = NimGame()
    game.play_game()

if __name__ == "__main__":
    run_nim_game()


WELCOME TO NIM GAME!
• The player who cannot make a move (all piles empty) loses
• Strategy tip: Watch the Nim-sum (XOR of pile sizes)!

Your turn!

Current piles: [5, 5, 5]
Pile 1: ***** (5)
Pile 2: ***** (5)
Pile 3: ***** (5)
Nim-sum: 5


Enter pile number (1-3):  2
Enter objects to remove (1-3):  2



You removed 2 objects from pile 2

Machine's turn...

Current piles: [5, 3, 5]
Pile 1: ***** (5)
Pile 2: *** (3)
Pile 3: ***** (5)
Nim-sum: 3
Machine removed 1 objects from pile 1

Your turn!

Current piles: [4, 3, 5]
Pile 1: **** (4)
Pile 2: *** (3)
Pile 3: ***** (5)
Nim-sum: 2


Enter pile number (1-3):  3
Enter objects to remove (1-3):  3



You removed 3 objects from pile 3

Machine's turn...

Current piles: [4, 3, 2]
Pile 1: **** (4)
Pile 2: *** (3)
Pile 3: ** (2)
Nim-sum: 5
Machine removed 1 objects from pile 1

Your turn!

Current piles: [3, 3, 2]
Pile 1: *** (3)
Pile 2: *** (3)
Pile 3: ** (2)
Nim-sum: 2


Enter pile number (1-3):  1
Enter objects to remove (1-3):  3



You removed 3 objects from pile 1

Machine's turn...

Current piles: [0, 3, 2]
Pile 1:  (0)
Pile 2: *** (3)
Pile 3: ** (2)
Nim-sum: 1
Machine removed 1 objects from pile 2

Your turn!

Current piles: [0, 2, 2]
Pile 1:  (0)
Pile 2: ** (2)
Pile 3: ** (2)
Nim-sum: 0


Enter pile number (1-3):  2
Enter objects to remove (1-3):  2



You removed 2 objects from pile 2

Machine's turn...

Current piles: [0, 0, 2]
Pile 1:  (0)
Pile 2:  (0)
Pile 3: ** (2)
Nim-sum: 2
Machine removed 2 objects from pile 3

GAME OVER!

Current piles: [0, 0, 0]
Pile 1:  (0)
Pile 2:  (0)
Pile 3:  (0)
Nim-sum: 0
Congratulations! You defeated the Machine!
